# 模块C-综合：ResNet18模型适配与分阶段微调

使用torchvision ResNet18替换分类头，先冻结骨干训练分类头，再解冻layer4微调并保存最终模型。

> 按 TODO 编号逐项补全。


In [ ]:
import cv2,pandas as pd,torch
from torch import nn,optim
from torch.utils.data import TensorDataset,DataLoader,random_split
from torchvision import models
from sklearn.metrics import classification_report
img=cv2.imread("shapes_grid.pgm",0);labels=pd.read_csv("shape_labels.csv").label.values;tiles=[img[r*16:(r+1)*16,c*16:(c+1)*16] for r in range(8) for c in range(20)];X=torch.tensor(tiles,dtype=torch.float32).unsqueeze(1)/255;X=torch.nn.functional.interpolate(X,size=(96,96)).repeat(1,3,1,1);y=torch.tensor(labels);ds=TensorDataset(X,y);train_ds,test_ds=random_split(ds,[128,32],generator=torch.Generator().manual_seed(42));train=DataLoader(train_ds,32,shuffle=True);test=DataLoader(test_ds,64)
# TODO 1-1：创建离线ResNet18
model=__________
# TODO 1-2：冻结全部骨干参数
for p in model.parameters():__________
# TODO 1-3：替换fc为4分类
model.fc=__________;loss_fn=nn.CrossEntropyLoss();opt=optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=1e-3)
for _ in range(3):
    for x,b in train:opt.zero_grad();loss=loss_fn(model(x),b);loss.backward();opt.step()
# TODO 2-1：解冻layer4微调
for p in model.layer4.parameters():__________
opt=optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=2e-4)
for _ in range(2):
    for x,b in train:opt.zero_grad();loss=loss_fn(model(x),b);loss.backward();opt.step()
ys=[];ps=[];model.eval()
with torch.no_grad():
    for x,b in test:ps+=model(x).argmax(1).tolist();ys+=b.tolist()
print(classification_report(ys,ps,zero_division=0));__________